In [ ]:
from glob import glob
from os import path
from datetime import datetime as dt, timedelta
import xarray as xr
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from goes2go.data import goes_nearesttime
from cartopy import crs as ccrs
from cartopy import feature as cfeat
from pyxlma import coords
import cmweather
import warnings
from metpy.units import units
from metpy import calc as mpcalc
from metpy import plots as mpplots
from geopandas import read_file

In [ ]:
sbf_fineline_ex_dt = dt(2022, 7, 14, 21, 50, 0)
goes_data = goes_nearesttime(sbf_fineline_ex_dt, satellite='goes16', product='ABI', return_as='xarray')
goes_data['TrueColor'] = goes_data.rgb.TrueColor()
imkw = goes_data.rgb.imshow_kwargs
imkw.pop('transform')

In [ ]:
radar_filepaths = sorted(glob(f'/Volumes/LtgSSD/nexrad_zarr/{sbf_fineline_ex_dt.strftime('%B').upper()}/{sbf_fineline_ex_dt.strftime('%Y%m%d')}/*'))
radar_times = np.array([dt.strptime(path.basename(f), 'KHGX%Y%m%d_%H%M%S_V06_grid.zarr') for f in radar_filepaths])
closest_radar_idx = np.argmin(np.abs(radar_times - sbf_fineline_ex_dt))
closest_radar_path = radar_filepaths[closest_radar_idx]
radar_data = xr.open_dataset(closest_radar_path, engine='zarr').isel(time=0, nradar=0)
tpcs_coords = coords.TangentPlaneCartesianSystem(ctrLat=float(radar_data.radar_latitude.data), ctrLon=float(radar_data.radar_longitude.data), ctrAlt=float(radar_data.radar_altitude.data))
geosys = coords.GeographicSystem()
x2d, y2d = np.meshgrid(radar_data.x.data, radar_data.y.data)
ecef_coords = tpcs_coords.toECEF(x2d.flatten(), y2d.flatten(), np.full_like(x2d, 1000).flatten())
radar_lon, radar_lat, _ = lla = geosys.fromECEF(*ecef_coords)
radar_lon.shape = x2d.shape
radar_lat.shape = y2d.shape

In [ ]:
madis_file = path.join(path.sep, 'Volumes', 'LtgSSD', 'sfcdata_madis', sbf_fineline_ex_dt.strftime('%Y%m%d_*'))
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    madis_ds = xr.open_mfdataset(madis_file, engine='netcdf4', chunks='auto', coords='minimal', concat_dim='recNum', combine='nested', compat='override')
madis_ds = madis_ds.where(((madis_ds.longitude <= radar_lon.max()) & (madis_ds.longitude >= radar_lon.min()) & (madis_ds.latitude <= radar_lat.max()) & (madis_ds.latitude >= radar_lat.min())).compute(), drop=True)
dims_to_rm = list(madis_ds.dims)
dims_to_rm.remove('recNum')
madis_ds = madis_ds.drop_dims(dims_to_rm)
madis_ds_temp = madis_ds.temperature.data
madis_ds_temp_qc = madis_ds.temperatureQCR.data

madis_ds_dew = madis_ds.dewpoint.data
madis_ds_dew_qc = madis_ds.dewpointQCR.data

madis_ds_time = madis_ds.observationTime.data
madis_ds_lat = madis_ds.latitude.data
madis_ds_lon = madis_ds.longitude.data

madis_ds_invalid = np.zeros_like(madis_ds_temp, dtype=bool)
madis_ds_invalid[((madis_ds_temp_qc != 0) | (madis_ds_dew_qc != 0) | np.isnan(madis_ds_temp) | np.isnan(madis_ds_dew)).compute()] = True

madis_ds_temp[madis_ds_invalid] = np.nan
madis_ds_temp = madis_ds_temp.compute()
madis_ds_dew[madis_ds_invalid] = np.nan
madis_ds_dew = madis_ds_dew.compute()
madis_ds_time[madis_ds_invalid] = np.datetime64('NaT')
madis_ds_time = madis_ds_time.astype('datetime64[s]').compute()
madis_ds_lat[madis_ds_invalid] = np.nan
madis_ds_lat = madis_ds_lat.compute()
madis_ds_lon[madis_ds_invalid] = np.nan
madis_ds_lon = madis_ds_lon.compute()

lower_time_bound = np.array([sbf_fineline_ex_dt-timedelta(hours=1)]).astype('datetime64[s]')[0]
upper_time_bound = np.array([sbf_fineline_ex_dt]).astype('datetime64[s]')[0]
before = (madis_ds.observationTime <= upper_time_bound).compute()
after = (madis_ds.observationTime >= lower_time_bound).compute()
rec_to_consider = madis_ds.where((before & after), drop=True)
df = rec_to_consider[['observationTime', 'stationId']].to_dataframe()
latest_records_idx = df.groupby('stationId')['observationTime'].idxmax()
latest_obs = rec_to_consider.sel(recNum=latest_records_idx)
dir = latest_obs.windDir.data.compute() * units.degrees
spd = (latest_obs.windSpeed.data.compute() * units.meter / units.second).to(units.knots)
u, v = mpcalc.wind_components(spd, dir)

In [ ]:
sbf_interp = read_file(f'sam_polyline/{sbf_fineline_ex_dt.strftime("%Y-%m-%d_interpolated.json")}').set_index('index', drop=True)
poly_i_want = sbf_interp[sbf_interp.index == radar_data.time.data.astype('datetime64[s]').astype(dt)]

In [ ]:
sbf_fineline_ex_fig = plt.figure(figsize=(10, 10))
sbf_fineline_ex_axs = sbf_fineline_ex_fig.subplots(2, 2, subplot_kw={'projection' : ccrs.LambertConformal()})
[ax.set_extent([-97, -93, 28, 31], crs=ccrs.PlateCarree()) for ax in sbf_fineline_ex_axs.flatten()]
sbf_fineline_ex_axs[0, 0].imshow(goes_data['TrueColor'].data, **imkw, transform=goes_data.rgb.crs)
sbf_fineline_ex_axs[0, 0].set_title(f'GOES-16 True Color\n{goes_data.time_bounds.min().astype("datetime64[s]").data.astype(dt).item().strftime("%Y-%m-%d %H:%M:%S")}')
rdr_ref = sbf_fineline_ex_axs[0, 1].pcolormesh(radar_lon, radar_lat, radar_data.reflectivity.sel(z=1000), vmin=-10, vmax=80, cmap='ChaseSpectral', transform=ccrs.PlateCarree(), rasterized=True)
sbf_fineline_ex_axs[0, 1].set_title(f'KHGX 1km AGL Radar Reflectivity\n{radar_data.time.data.astype("datetime64[s]").astype(dt).item().strftime("%Y-%m-%d %H:%M:%S")}')
sbf_fineline_ex_fig.colorbar(rdr_ref, ax=sbf_fineline_ex_axs[0, 1], orientation='vertical', label='Reflectivity (dBZ)')
[ax.add_feature(cfeat.COASTLINE) for ax in sbf_fineline_ex_axs.flatten()]
stations = mpplots.StationPlot(sbf_fineline_ex_axs[1, 0], latest_obs.longitude, latest_obs.latitude, clip_on=True, transform=ccrs.PlateCarree(), fontsize=6)
stations.plot_barb(u, v, sizes={"emptybarb" : 0}, zorder=2)
sbf_fineline_ex_axs[1, 0].set_title(f'MADIS Surface Wind Barbs (kt)\n{sbf_fineline_ex_dt.strftime("%Y-%m-%d %H:%M")}')

sbf_fineline_ex_axs[1, 1].set_facecolor('tab:red')
poly_i_want.plot(ax=sbf_fineline_ex_axs[1, 1], color='tab:blue', linewidth=1, transform=ccrs.PlateCarree())
sbf_fineline_ex_axs[1, 1].set_title('Subjectively Analyzed Seabreeze Front\n(Red continental airmass, Blue maritime airmass)')

sbf_fineline_ex_fig.suptitle(f'Seabreeze Identification Example')
sbf_fineline_ex_fig.tight_layout()

sbf_fineline_ex_fig.savefig(f'./thesis_figs/seabreeze_bdy_example.pdf')